构建候选集

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import bisect

tqdm.pandas(desc="building history")   # 激活 pandas 进度条

In [2]:
df1 = pd.read_csv(r"D:\rec\project\Kuairand\KuaiRand-1K\KuaiRand-1K\data\log_standard_4_08_to_4_21_1k.csv")
df2 = pd.read_csv(r"D:\rec\project\Kuairand\KuaiRand-1K\KuaiRand-1K\data\log_standard_4_22_to_5_08_1k.csv")
df = pd.concat([df1, df2], ignore_index=True)
del df1, df2

labels = ['is_click', 'is_like', 'is_comment', 'is_follow', 'is_forward','long_view']
unuse_cols = ['is_hate', 'play_time_ms', 'duration_ms', 'profile_stay_time','comment_stay_time', 'is_profile_enter','is_rand', 'tab']
df = df.drop(unuse_cols, axis=1)

df.columns

Index(['user_id', 'video_id', 'date', 'hourmin', 'time_ms', 'is_click',
       'is_like', 'is_follow', 'is_comment', 'is_forward', 'long_view'],
      dtype='object')

In [3]:
df.head(2)

,user_id,video_id,date,hourmin,time_ms,is_click,is_like,is_follow,is_comment,is_forward,long_view
0,0,4354972,20220409,900,1649467982289,0,0,0,0,0,0
1,0,1329429,20220409,900,1649467982289,0,0,0,0,0,0


In [3]:
# 1. 先把 time_ms 转 datetime，方便取「首条时间」做对齐
df['date'] = pd.to_datetime(df['date'], format='%Y%m%d')   # 2022-04-09

hh = df['hourmin'] // 100
mm = df['hourmin'] % 100

df['dt'] = pd.to_datetime(
    df['date'].dt.strftime('%Y-%m-%d') + ' '
    + hh.astype(str).str.zfill(2) + ':' + mm.astype(str).str.zfill(2),
    format='%Y-%m-%d %H:%M'
)                 

# 2. 为每个用户计算「首条记录时间」并向下取到整点，作为该用户的 0 号 session 起点
user_base = (df.groupby('user_id')['dt']
               .min()
               .dt.floor('h')          # 整点
               .rename('base_time'))

df = df.merge(user_base, left_on='user_id', right_index=True)

# 3. 计算每条记录相对于 base_time 的小时偏移 → session 编号
df['session_sn'] = (df['dt'] - df['base_time']).dt.total_seconds() // 3600
df['session_id'] = df['user_id'].astype(str) + '_' + df['session_sn'].astype(int).astype(str)
df['timestamp_s'] = (df['time_ms'] // 1000).astype(int)

# 4. 把同一 session 内的所有记录聚合成列表，并生成 session_start
cols = ['video_id', 'is_click', 'is_like', 'is_follow',
        'is_comment', 'is_forward', 'long_view', 'timestamp_s', 'dt']

session_df = (df.groupby(['session_id','user_id'])[cols]
                .agg({'video_id': list,
                      'is_click': list,
                      'is_like': list,
                      'is_follow': list,
                      'is_comment': list,
                      'is_forward': list,
                      'long_view': list,
                      'timestamp_s': list,
                      'dt': 'min'})          #  取 session 内最早时间
                .rename(columns={'dt': 'session_start'})
                .reset_index())

# 5. 重新命名
new_df = session_df[['session_id', 'user_id', 'session_start','timestamp_s',
                     'video_id', 'is_click', 'is_like', 'is_follow',
                     'is_comment', 'is_forward', 'long_view']]

# 6. 筛除交互记录过少和过多点击列表全为0的 session

filtered_df = new_df[
    (new_df['is_click'].apply(np.sum) > 20)
].copy()

In [4]:
filtered_df.head(2)

,session_id,user_id,session_start,timestamp_s,video_id,is_click,is_like,is_follow,is_comment,is_forward,long_view
18,0_263,0,2022-04-20 08:00:00,"[1650412408, 1650412408, 1650412408, 165041240...","[2542388, 1517347, 542968, 3356692, 229714, 38...","[1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, ..."
40,0_311,0,2022-04-22 08:00:00,"[1650585311, 1650585311, 1650585311, 165058531...","[2019165, 2133392, 3800048, 345849, 1033272, 1...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."


构造用户历史特征

In [5]:
df1 = pd.read_csv(r"D:\rec\project\Kuairand\KuaiRand-1K\KuaiRand-1K\data\log_standard_4_08_to_4_21_1k.csv")
df2 = pd.read_csv(r"D:\rec\project\Kuairand\KuaiRand-1K\KuaiRand-1K\data\log_standard_4_22_to_5_08_1k.csv")
df = pd.concat([df1, df2], ignore_index=True)
del df1, df2

filtered_df['session_start'] = (pd.to_datetime(filtered_df['session_start'])
                            .astype('int64') // 10**6).astype('int64') 

In [6]:
def get_history(row, max_len=30):
    uid, t = row['user_id'], row['session_start']
    u_df = user_group.get_group(uid)
    click_history = u_df[(u_df['time_ms'] < t) & (u_df['is_click'] == 1)]
    recent = click_history['video_id'].tail(max_len).tolist()
    return pd.Series({'video_his': recent, 'len_his': len(recent)})


In [7]:
user_group = df.groupby('user_id')            # 先建组，避免每次重新 group
hist_cols = filtered_df.progress_apply(get_history, axis=1)
filtered_df = pd.concat([filtered_df, hist_cols], axis=1)

filtered_df['session_start'] = pd.to_datetime(filtered_df['session_start'], unit='ms')
filtered_df = filtered_df.sort_values('session_start').reset_index(drop=True)

building history: 100%|██████████| 78918/78918 [01:55<00:00, 683.05it/s] 


In [8]:
filtered_df.head(2)

,session_id,user_id,session_start,timestamp_s,video_id,is_click,is_like,is_follow,is_comment,is_forward,long_view,video_his,len_his
0,324_0,324,2022-04-08,"[1649346741, 1649346741, 1649346741, 164934674...","[3225716, 750958, 1561681, 2848594, 3942552, 9...","[1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, ...","[4354080, 2217794, 294865, 2444848, 1170985, 1...",30
1,7_0,7,2022-04-08,"[1649346604, 1649346734, 1649346734, 164934869...","[3283168, 795137, 453155, 3352672, 855058, 916...","[0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, ...","[3121181, 3648619, 4025106, 2751137, 4240524, ...",30


In [9]:
counts = filtered_df['video_his'].apply(len).value_counts()
print(counts)


video_his
30    78884
28        6
22        5
29        5
24        4
23        4
26        4
21        2
25        2
27        2
Name: count, dtype: int64


检查数据是否正确

In [10]:
cols = ['video_id', 'is_click', 'is_like', 'is_follow',
        'is_comment', 'is_forward', 'long_view']
bad = []
for idx, row in filtered_df.iterrows():
    lengths = {
        col: len(row[col]) if isinstance(row[col], list) else None
        for col in cols
    }
    valid_lengths = {v for v in lengths.values() if v is not None}
    if not valid_lengths:
        continue
    if len(valid_lengths) > 1:
        bad.append({'index': idx, 'session_id': row.get('session_id'),
                    'lengths': lengths})
bad

[]

划分数据集

In [11]:
train_len = int(len(filtered_df) * 0.8)
train_df = filtered_df[:train_len]
tmp_df = filtered_df[train_len:]
val_df = tmp_df[:int(len(tmp_df)/2)]
test_df = tmp_df[int(len(tmp_df)/2):]

In [13]:
train_df.to_csv('train.csv', index=False, encoding='utf-8')
val_df.to_csv('val.csv', index=False, encoding='utf-8')
test_df.to_csv('test.csv', index=False, encoding='utf-8')

In [17]:
train_df.head(1).timestamp_s.to_list()

[[1649346741,
  1649346741,
  1649346741,
  1649346741,
  1649346741,
  1649346741,
  1649347248,
  1649347248,
  1649347248,
  1649347248,
  1649347248,
  1649347248,
  1649347443,
  1649347443,
  1649347443,
  1649347443,
  1649347443,
  1649347443,
  1649347443,
  1649347640,
  1649347640,
  1649347640,
  1649347640,
  1649347640,
  1649347640,
  1649347772,
  1649347772,
  1649347772,
  1649347772,
  1649347772,
  1649347772,
  1649347772,
  1649347978,
  1649347978,
  1649347978,
  1649347978,
  1649347978,
  1649348259,
  1649348259,
  1649348259,
  1649348259,
  1649348259,
  1649348259,
  1649348426,
  1649348426,
  1649348426,
  1649348426,
  1649348426,
  1649348426,
  1649349087,
  1649349087,
  1649349087,
  1649349087,
  1649349087,
  1649349087,
  1649349303,
  1649349303,
  1649349303,
  1649349303,
  1649349303,
  1649349303,
  1649349303,
  1649349613,
  1649349613,
  1649349613,
  1649349613,
  1649349613,
  1649349613,
  1649349709]]

简要数据分析

In [13]:
# 1. 用户/视频规模 & 总交互数
n_users  = filtered_df['user_id'].nunique()
n_videos = len(np.unique(np.concatenate(filtered_df['video_id'].tolist())))

print('===== 基本规模 =====')
print(f'用户数 (user_id) : {n_users:,}')
print(f'物品数 (video_id): {n_videos:,}')
print(f'总session记录数     : {filtered_df.shape[0]}')

# 2. 六种行为的正负比例
behavs = ['is_like', 'is_comment', 'is_follow',
          'is_forward', 'long_view']

print('\n===== 行为正负样本比例 =====')

col = 'is_click'
flat = pd.Series(np.concatenate(filtered_df[col].values))
pos  = flat.sum()
tot  = len(flat)
neg  = tot - pos
print(f'{col:12}  正: {pos:>8,}  负: {neg:>8,}  正样本占比: {pos/tot:6.2%}, 负正样本比例: {neg/pos:.3f}')

click_cnt = tot
for col in behavs:
    flat = pd.Series(np.concatenate(filtered_df[col].values))
    pos  = flat.sum()
    neg  = click_cnt - pos
    print(f'{col:12}  正: {pos:>8,}  负: {neg:>8,}  正样本占比: {pos/click_cnt:6.2%}, 负正样本比例: {neg/pos:.3f}')

===== 基本规模 =====
用户数 (user_id) : 994
物品数 (video_id): 3,338,293
总session记录数     : 78918

===== 行为正负样本比例 =====
is_click      正: 3,403,831  负: 4,902,496  正样本占比: 40.98%, 负正样本比例: 1.440
is_like       正:  130,052  负: 8,176,275  正样本占比:  1.57%, 负正样本比例: 62.869
is_comment    正:   22,466  负: 8,283,861  正样本占比:  0.27%, 负正样本比例: 368.729
is_follow     正:    7,509  负: 8,298,818  正样本占比:  0.09%, 负正样本比例: 1105.183
is_forward    正:    6,312  负: 8,300,015  正样本占比:  0.08%, 负正样本比例: 1314.958
long_view     正: 2,347,031  负: 5,959,296  正样本占比: 28.26%, 负正样本比例: 2.539


In [13]:
# 1. 用户级冷启动
test_users  = set(test_df['user_id'])
train_users = set(train_df['user_id'])
cold_users  = test_users - train_users
print('===== 用户冷启动 =====')
print(f'测试集用户数     : {len(test_users):,}')
print(f'新用户数         : {len(cold_users):,}')
print(f'用户冷启动比例   : {len(cold_users)/len(test_users):6.2%}')

# 2. 物品级冷启动
train_items = set(np.concatenate(train_df['video_id'].tolist()))
test_items  = set(np.concatenate(test_df['video_id'].tolist()))
cold_items  = test_items - train_items
print('\n===== 物品冷启动 =====')
print(f'测试集物品数     : {len(test_items):,}')
print(f'新物品数         : {len(cold_items):,}')
print(f'物品冷启动比例   : {len(cold_items)/len(test_items):6.2%}')

===== 用户冷启动 =====
测试集用户数     : 898
新用户数         : 6
用户冷启动比例   :  0.67%

===== 物品冷启动 =====
测试集物品数     : 481,659
新物品数         : 342,273
物品冷启动比例   : 71.06%
